# Copper Canyon — Download original website images to Google Drive

**What this does:** Downloads public original-resolution photo, logo, and graphic files from [thecoppercanyon.com](https://www.thecoppercanyon.com/) and stores the actual image files in **Copper Canyon Website Assets → Original Website Images**, with a source/verification CSV.

**How to run:** From the **Runtime** menu choose **Run all**. When asked, authorize **tucker.t.brennan@gmail.com**. The notebook checks that the destination folder exists before writing anything.

Uses Google Colab (free tier is sufficient for this small crawl), Python, and the [crawler source in the Copper Foods GitHub repository](https://github.com/tuckerbrennan/copper-foods-website/blob/main/scripts/archive_copper_canyon_assets.py). It doesn't use Canva and doesn't require a local terminal.

**Important:** Downloading can take a few minutes. Photos are fetched from Wix's public CDN and some may fail; check `image_manifest.csv`. This is a public-site archive, not an export of private Wix Media Manager assets. Obtain appropriate photo reuse permission from Michael before using the images in production.


In [ ]:
# Run this cell and choose tucker.t.brennan@gmail.com when Google asks which account to authorize.
from pathlib import Path
import subprocess
import sys
import shutil
import requests
from google.colab import drive

drive.mount("/content/drive")

# Require the previously created folder in the intended Google Drive account.
# Deliberately DO NOT create a duplicate if you mounted the wrong account.
destination_root = Path("/content/drive/MyDrive/Copper Canyon Website Assets")
if not destination_root.is_dir():
    raise RuntimeError(
        "I cannot find 'Copper Canyon Website Assets' in the mounted account. "
        "Disconnect Drive and mount tucker.t.brennan@gmail.com; "
        "do not proceed with the wrong account."
    )

print("Installing the small set of crawler dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "-q", "install", "requests", "beautifulsoup4"],
    check=True
)

url = (
    "https://raw.githubusercontent.com/tuckerbrennan/"
    "copper-foods-website/main/scripts/archive_copper_canyon_assets.py"
)
response = requests.get(url, timeout=30)
response.raise_for_status()
script_file = Path("/content/archive_copper_canyon_assets.py")
script_file.write_bytes(response.content)

print("Crawling the original Copper Canyon public site and downloading image files...")
result = subprocess.run([sys.executable, str(script_file)], cwd="/content", check=False)

archive = Path("/content/copper-canyon-site-archive")
destination = destination_root / "Original Website Images"
if not archive.is_dir():
    raise RuntimeError("The crawler did not create an archive. Check the log above.")

# Includes images, source URLs, checksums, failures and crawled page references.
shutil.copytree(archive, destination, dirs_exist_ok=True)

image_count = len([p for p in (destination / "images").iterdir() if p.is_file()])
print(f"Saved {image_count} original image files.")
print(f"Google Drive destination: {destination}")
print("Open 'image_manifest.csv' for a breakdown of successful and failed downloads.")
if result.returncode or not image_count:
    print("WARNING: Some or all remote image downloads failed. Review image_manifest.csv.")
else:
    print("Archive complete — image bytes are stored in Google Drive.")
